In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from scipy import stats

In [2]:
# Leitura dos Dados
df = pd.read_parquet(r'..\data\processed\bank_features.parquet')

# Configuração para visualizar todas as variávels
pd.set_option('display.max_columns', None)

In [3]:
df.shape

(41188, 27)

In [4]:
# Transformando a variável TARGET (y) em binária ANTES da separação
df['y'] = np.where(df['y'] == 'yes', 1, 0)

In [5]:
# Utilização de 70% para Treino
abt_01, abt_01_test = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df['y'] # Devido a desbalanceamento da variável TARGET
)

In [ ]:
# Excluindo variável TARGET do treino
abt_01 = abt_01.drop(columns=['y'])

# Excluindo a variável TARGET do Teste
abt_01_test = abt_01_test.drop(columns=['y'])

In [7]:
# Validação das bases
print(f'Formato da Memória: {type(abt_01)}')
print(f'Linhas do treino: {abt_01.shape[0]}')
print(f'Linhas do teste: {abt_01_test.shape[0]}')

Formato da Memória: <class 'pandas.DataFrame'>
Linhas do treino: 28831
Linhas do teste: 12357


# Metadados

In [8]:
def visualizar_metadados(df) -> pd.DataFrame:
    """
    Função para visualizar Metadados de um Dataframe.
    return: Tipo, Cardinalidade, Qtd de Nulos e Percent Nulos
    param: Dataframe fornecido para análise
    """

    metadata = pd.DataFrame({
        'Nome': df.columns,
        'Tipo': df.dtypes,
        'Cardinalidade': df.nunique(),
        'Qtd_Nulos': df.isnull().sum(),
        'Percent_Nulos': round((df.isnull().sum() / len(df)) * 100, 2),
    })

    # Função para verificar o teste de Normalidade (Shapiro-Wilk)
    def test_normalidade(series,alpha=0.05):
        if series.dtypes in ['float64', 'int64']:
            statistic, pvalue = stats.shapiro(series.dropna())
            return pvalue > alpha
        else:
            return None

    # Aplicando o teste de Normalidade
    metadata['fl_normal'] = df.apply(test_normalidade)

    metadata = metadata.sort_values(by='Percent_Nulos', ascending=False)
    metadata = metadata.reset_index(drop=True)

    return metadata

In [9]:
visualizar_metadados(abt_01)

c:\Users\felip\PythonProjects\Projetos_Pod_Academy\bank-marketing\venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 28831.
  res = hypotest_fun_out(*samples, **kwds)


,Nome,Tipo,Cardinalidade,Qtd_Nulos,Percent_Nulos,fl_normal
0,age,int64,74,0,0.0,False
1,job,str,12,0,0.0,None
2,marital,str,4,0,0.0,None
3,education,str,8,0,0.0,None
4,default,str,3,0,0.0,None
5,housing,str,3,0,0.0,None
6,loan,str,3,0,0.0,None
7,contact,str,2,0,0.0,None
8,month,str,10,0,0.0,None
9,day_of_week,str,5,0,0.0,None


In [10]:
metadata_df = visualizar_metadados(abt_01)

c:\Users\felip\PythonProjects\Projetos_Pod_Academy\bank-marketing\venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 28831.
  res = hypotest_fun_out(*samples, **kwds)


# Tratamento das Variáveis Numéricas

In [11]:
# Copiando para utilizar tratamentos diferentes.
abt_02 = abt_01.copy()
abt_03 = abt_01.copy()

## 1. RobustScaler (Mediana e IQR)

In [12]:
scaler_robust = RobustScaler()

# Selecionando as colunas numéricas
cols_numericas = abt_01.select_dtypes(include=['float64', 'int64', 'int32']).columns

# Escalador Robusto - (Mediana e IQR)
abt_01[cols_numericas] = scaler_robust.fit_transform(abt_01[cols_numericas])
abt_01.shape

(28831, 26)

In [13]:
abt_01.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,flg_contacted_prev,total_credits_active,total_contacts,ratio_campaign_previous,economic_stress_index,month_int,months_since_campaign
7309,0.800000,blue-collar,single,basic.4y,no,no,no,telephone,may,thu,0.0,0.0,0.0,nonexistent,0.00000,0.266594,0.857143,0.000829,0.000000,0.0,-1.0,0.0,0.0,0.000359,-0.333333,0.333333
4169,0.866667,services,married,professional.course,unknown,no,no,telephone,may,mon,1.5,0.0,0.0,nonexistent,0.00000,0.266594,0.857143,0.000276,0.000000,0.0,-1.0,1.5,1.5,0.000120,-0.333333,0.333333
35810,0.533333,technician,divorced,professional.course,no,yes,no,cellular,may,mon,0.0,0.0,0.0,nonexistent,-0.90625,-0.931447,-0.698413,-0.998894,-0.712403,0.0,0.0,0.0,0.0,-0.824532,-0.333333,0.333333
9668,0.533333,admin.,single,university.degree,no,yes,no,telephone,jun,mon,-0.5,0.0,0.0,nonexistent,0.09375,0.779108,0.000000,0.028753,0.287597,0.0,0.0,-0.5,-0.5,0.174294,0.000000,0.000000
34389,-0.866667,services,single,high.school,no,no,no,cellular,may,thu,1.5,0.0,0.0,nonexistent,-0.90625,-0.931447,-0.698413,-0.992812,-0.712403,0.0,-1.0,1.5,1.5,-0.828838,-0.333333,0.333333


## 2. Padronização (Média e Desvio Padrão)

In [14]:
scaler_std = StandardScaler()

# Padronização (Média e Desvio Padrão)
abt_02[cols_numericas] = scaler_std.fit_transform(abt_02[cols_numericas])
abt_02.shape

(28831, 26)

In [15]:
abt_02.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,flg_contacted_prev,total_credits_active,total_contacts,ratio_campaign_previous,economic_stress_index,month_int,months_since_campaign
7309,0.961016,blue-collar,single,basic.4y,no,no,no,telephone,may,thu,-0.202499,-0.172526,-0.350844,nonexistent,0.648136,0.722717,0.887472,0.714873,0.332204,-0.198883,-1.071154,-0.265925,-0.192089,0.572598,-0.791161,0.791161
4169,1.057229,services,married,professional.course,unknown,no,no,telephone,may,mon,0.887201,-0.172526,-0.350844,nonexistent,0.648136,0.722717,0.887472,0.713720,0.332204,-0.198883,-1.071154,0.821626,0.895376,0.572072,-0.791161,0.791161
35810,0.576165,technician,divorced,professional.course,no,yes,no,cellular,may,mon,-0.202499,-0.172526,-0.350844,nonexistent,-1.198744,-1.181025,-1.231784,-1.368338,-0.935879,-0.198883,0.509804,-0.265925,-0.192089,-1.238277,-0.791161,0.791161
9668,0.576165,admin.,single,university.degree,no,yes,no,telephone,jun,mon,-0.565732,-0.172526,-0.350844,nonexistent,0.839193,1.537124,-0.280281,0.773060,0.844128,-0.198883,0.509804,-0.628441,-0.554577,0.954435,-0.300117,0.300117
34389,-1.444302,services,single,high.school,no,no,no,cellular,may,thu,0.887201,-0.172526,-0.350844,nonexistent,-1.198744,-1.181025,-1.231784,-1.355664,-0.935879,-0.198883,-1.071154,0.821626,0.895376,-1.247731,-0.791161,0.791161


## 3. Normalização (Min-Max)

In [16]:
scaler_minmax = MinMaxScaler()

# Normalização (Min-Max)
abt_03[cols_numericas] = scaler_minmax.fit_transform(abt_03[cols_numericas])
abt_03.shape

(28831, 26)

In [17]:
abt_03.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,flg_contacted_prev,total_credits_active,total_contacts,ratio_campaign_previous,economic_stress_index,month_int,months_since_campaign
7309,0.407407,blue-collar,single,basic.4y,no,no,no,telephone,may,thu,0.018182,0.0,0.0,nonexistent,0.937500,0.698753,0.602510,0.958059,0.859735,0.0,0.0,0.018182,0.033298,0.849168,0.222222,0.777778
4169,0.419753,services,married,professional.course,unknown,no,no,telephone,may,mon,0.072727,0.0,0.0,nonexistent,0.937500,0.698753,0.602510,0.957606,0.859735,0.0,0.0,0.072727,0.087003,0.848962,0.222222,0.777778
35810,0.358025,technician,divorced,professional.course,no,yes,no,cellular,may,mon,0.018182,0.0,0.0,nonexistent,0.333333,0.269680,0.192469,0.138291,0.512287,0.0,0.5,0.018182,0.033298,0.139435,0.222222,0.777778
9668,0.358025,admin.,single,university.degree,no,yes,no,telephone,jun,mon,0.000000,0.0,0.0,nonexistent,1.000000,0.882307,0.376569,0.980957,1.000000,0.0,0.5,0.000000,0.015396,0.998821,0.333333,0.666667
34389,0.098765,services,single,high.school,no,no,no,cellular,may,thu,0.072727,0.0,0.0,nonexistent,0.333333,0.269680,0.192469,0.143278,0.512287,0.0,0.0,0.072727,0.087003,0.135730,0.222222,0.777778


# Tratamento das Variáveis Categóricas

In [ ]:
card_cutoff = 15

# Selecionando colunas String com Cardinalidade Igual ou Menor a 15.
df_cat_onehot = metadata_df[(metadata_df['Cardinalidade'] <= card_cutoff) & (metadata_df['Tipo'] == 'str')]
lista_onehot = list(df_cat_onehot.Nome.values)
print('Lista com as variáveis do OneHot: ', lista_onehot)

# Excluir a variável Education, para utilizar com OrdinalEncoding
eduction_removido = lista_onehot.pop(2)
print('Lista atualizada do OneHot: ', lista_onehot)

# drop = first -> Evita multicolinariedade em variáveis binárias
# sparse_output -> retorna um array denso que o pandas consegue ler
# handle_unknown = 'ignore' -> vai ignorar novas categorias acrescentadas no treino.
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# Transformador aprende as categorias e gera um array codificado
enconded_data = encoder.fit_transform(abt_01[lista_onehot])

# get_feature_names_out() extrai os nomes
encoded_cols = encoder.get_feature_names_out(lista_onehot)

# Criando um DataFrame com as colunas codificadas, mantando o índice original.
df_train_ohe = pd.DataFrame(enconded_data, columns=encoded_cols, index=abt_01.index)

# Concatenando o DataFrame original com o DataFrame codificado.
abt_01 = pd.concat([abt_01.drop(lista_onehot, axis=1), df_train_ohe], axis=1)

print(f'Número de colunas originais: {abt_01.shape[1]}')
display(abt_01.head())

Lista com as variáveis do OneHot:  ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
Lista atualizada do OneHot:  ['job', 'marital', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
Número de colunas originais: 53


,age,education,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,flg_contacted_prev,total_credits_active,total_contacts,ratio_campaign_previous,economic_stress_index,month_int,months_since_campaign,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,job_unknown,marital_married,marital_single,marital_unknown,default_unknown,default_yes,housing_unknown,housing_yes,loan_unknown,loan_yes,contact_telephone,month_aug,month_dec,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
7309,0.800000,basic.4y,0.0,0.0,0.0,0.00000,0.266594,0.857143,0.000829,0.000000,0.0,-1.0,0.0,0.0,0.000359,-0.333333,0.333333,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4169,0.866667,professional.course,1.5,0.0,0.0,0.00000,0.266594,0.857143,0.000276,0.000000,0.0,-1.0,1.5,1.5,0.000120,-0.333333,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
35810,0.533333,professional.course,0.0,0.0,0.0,-0.90625,-0.931447,-0.698413,-0.998894,-0.712403,0.0,0.0,0.0,0.0,-0.824532,-0.333333,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
9668,0.533333,university.degree,-0.5,0.0,0.0,0.09375,0.779108,0.000000,0.028753,0.287597,0.0,0.0,-0.5,-0.5,0.174294,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
34389,-0.866667,high.school,1.5,0.0,0.0,-0.90625,-0.931447,-0.698413,-0.992812,-0.712403,0.0,-1.0,1.5,1.5,-0.828838,-0.333333,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [21]:
# Definindo a hierarquia explícita da menor escolaridade para a maior.

ordem_educacao = [
    'illiterate',          # 0: Analfabeto
    'basic.4y',            # 1: 4 anos de estudo
    'basic.6y',            # 2: 6 anos de estudo
    'basic.9y',            # 3: 9 anos de estudo
    'high.school',         # 4: Ensino Médio
    'professional.course', # 5: Curso Profissionalizante/Técnico
    'university.degree',   # 6: Ensino Superior
    'unknown'              # 7: Desconhecido (topo/fim para não misturar com o resto)
]

# Inicializando o OrdinalEncoder passando a lista de categorias dentro de outra lista
encoder_ord = OrdinalEncoder(categories=[ordem_educacao], handle_unknown='use_encoded_value', unknown_value=-1)

# [['education']] <- porque o scikit-learn espera um Dataframe 2D
abt_01['education'] = encoder_ord.fit_transform(abt_01[['education']])

print('Valores transformando em Hierarquia:')
display(abt_01['education'].value_counts())


Valores transformando em Hierarquia:


education
6.0    8518
4.0    6690
3.0    4218
5.0    3679
1.0    2865
2.0    1615
7.0    1234
0.0      12
Name: count, dtype: int64

In [ ]:
# Visualizando a tabela pós Data Prep
abt_01.head()

,age,education,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,flg_contacted_prev,total_credits_active,total_contacts,ratio_campaign_previous,economic_stress_index,month_int,months_since_campaign,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,job_unknown,marital_married,marital_single,marital_unknown,default_unknown,default_yes,housing_unknown,housing_yes,loan_unknown,loan_yes,contact_telephone,month_aug,month_dec,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
7309,0.800000,1.0,0.0,0.0,0.0,0.00000,0.266594,0.857143,0.000829,0.000000,0.0,-1.0,0.0,0.0,0.000359,-0.333333,0.333333,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4169,0.866667,5.0,1.5,0.0,0.0,0.00000,0.266594,0.857143,0.000276,0.000000,0.0,-1.0,1.5,1.5,0.000120,-0.333333,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
35810,0.533333,5.0,0.0,0.0,0.0,-0.90625,-0.931447,-0.698413,-0.998894,-0.712403,0.0,0.0,0.0,0.0,-0.824532,-0.333333,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
9668,0.533333,6.0,-0.5,0.0,0.0,0.09375,0.779108,0.000000,0.028753,0.287597,0.0,0.0,-0.5,-0.5,0.174294,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
34389,-0.866667,4.0,1.5,0.0,0.0,-0.90625,-0.931447,-0.698413,-0.992812,-0.712403,0.0,-1.0,1.5,1.5,-0.828838,-0.333333,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [24]:
# Aplicando todas as transformações na base de Test.

# 1. Aplicando RobustScaler no Teste
abt_01_test[cols_numericas] = scaler_robust.transform(abt_01_test[cols_numericas])

# 2. Aplicando OneHotEncoding no Teste
encoded_data_test = encoder.transform(abt_01_test[lista_onehot])

# Recriando o DataFrame com as colunas codificadas
df_test_one = pd.DataFrame(encoded_data_test, columns=encoded_cols, index=abt_01_test.index)

# Concatenando e dropando as originais
abt_01_test = pd.concat([abt_01_test.drop(lista_onehot, axis=1), df_test_one], axis=1)

# 3. Aplicando o OrdinalEncoding na variável Education
abt_01_test['education'] = encoder_ord.transform(abt_01_test[['education']])

# Validação Final
print(f'Colunas no Treino: {abt_01.shape[1]}')
print(f'Colunas no Teste: {abt_01_test.shape[1]}')

Colunas no Treino: 53
Colunas no Teste: 54


In [25]:
abt_01_test.head()

,age,education,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,flg_contacted_prev,total_credits_active,total_contacts,ratio_campaign_previous,economic_stress_index,month_int,months_since_campaign,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,job_unknown,marital_married,marital_single,marital_unknown,default_unknown,default_yes,housing_unknown,housing_yes,loan_unknown,loan_yes,contact_telephone,month_aug,month_dec,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
9923,0.733333,7.0,-0.5,0.0,0.0,0.09375,0.779108,0.000000,0.028200,0.287597,0,0.0,-1.0,-0.5,-0.5,0.173989,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3477,1.200000,3.0,0.0,0.0,0.0,0.00000,0.266594,0.857143,0.000829,0.000000,0,0.0,0.0,0.0,0.0,0.000359,-0.333333,0.333333,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
12077,-0.200000,5.0,-0.5,0.0,0.0,0.09375,0.779108,0.000000,0.024882,0.287597,0,0.0,-1.0,-0.5,-0.5,0.172162,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1579,-0.400000,3.0,0.0,0.0,0.0,0.00000,0.266594,0.857143,-0.000553,0.000000,0,0.0,-1.0,0.0,0.0,-0.000239,-0.333333,0.333333,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
39993,-0.733333,6.0,1.0,4.0,2.0,-0.87500,0.332971,0.317460,-1.130771,-1.545736,1,1.0,0.0,2.0,0.0,-0.722818,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


In [ ]:
# Salvando a base depois de realizar o Data Prep
abt_01.to_parquet(r'..\data\raw\bank_features')